# New QE — one QP per Kaggle notebook

Attach `anginhok/enhance-for-human-mfqe-vvc-trainval-qp22-37`, enable GPU and Internet. Change only `QP` below to 22, 27, 32 or 37. This notebook trains one separate model for 300,000 successful updates with Charbonnier-only loss and best validation PSNR-Y selection.

Validation uses the first 30 frames per test18 sequence; the bundle excludes full test18.


In [ ]:
QP = 22  # 22, 27, 32 or 37

import os
import subprocess
import sys

def run(*args):
    subprocess.run([str(x) for x in args], check=True)

run("git", "clone", "https://github.com/dangdinh17/Enhance-for-Human.git",
    "/kaggle/working/Enhance-for-Human")
os.chdir("/kaggle/working/Enhance-for-Human")
run(sys.executable, "-m", "pip", "install", "-r", "requirements.txt")
run(sys.executable, "-m", "pip", "install", "-e", "tools/researchctl")
run(sys.executable, "-B", "scripts/bootstrap_kaggle.py")

CAMPAIGN = f"newqe_kaggle_qp{QP}_charb_300k_v1"
def ctl(*args):
    run(sys.executable, "-m", "researchctl.cli", "--profile", "kaggle", *args)


## Check and bounded smoke
Two diagnostic updates for the selected QP only; no main training yet.


In [ ]:
ctl("status", "--campaign", CAMPAIGN)
ctl("validate")
ctl("plan", "--campaign", CAMPAIGN)
ctl("preflight", "--campaign", CAMPAIGN)
ctl("run", "--campaign", CAMPAIGN, "--until", "smoke")


## Approve and train this QP
Run this cell when ready for the 300k main run. Bootstrap verifies the full dataset bundle; only the selected QP is trained.


In [ ]:
ctl("approve", "--campaign", CAMPAIGN, "--stage", "main")
ctl("run", "--campaign", CAMPAIGN, "--until", "main")


## Save and resume
Checkpoints and CSV logs: `runs/research/newqe_kaggle_qp{QP}_charb_300k_v1/QP{QP}/`. Save the entire `runs/` directory before ending the session. Restore it at the same project path with the same code/dependencies, rerun setup, then use `ctl("resume", "--campaign", CAMPAIGN, "--until", "main")`. Environment changes can require renewed preflight, smoke and approval.
